In [ ]:
import sys
import os

# 1. DRIVE MOUNT AND PATHS SETUP
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory: {os.getcwd()}")
except ImportError:
    # Fallback for local execution
    DRIVE_ROOT = '.'
    print("Local execution")

In [ ]:
"""
Convert the per-video CSV logs written by the action recognition script into a log with
non-overlapping action intervals, one row per action and one object per row for each object type.

Input  (one row per sliding window and per person, objects marked 1 = associated to the person):
    Timestamp, person_1000, person_1001, plate_1000, plate_1001, cup_1001, ..., Action
    0.00s-2.00s, 1, 0, 1, 1, 0, ..., Reaching; CarryingWhileLocomoting

Output (one row per action interval and, when needed, per object):
    Start, End, Person, Cup, Plate, ..., Action
    0.00, 2.50, person_1000, cup_1001, plate_1000, Reaching
    0.00, 2.50, person_1000,          , plate_1001, Reaching

- Start / End (seconds): every window "owns" the time between its midpoint and the midpoints of
  the previous / next window (never outside the window itself), so the time owned by different
  windows never overlaps. Successive windows of the same person with the same action are merged
  into one row going from the start of the first to the end of the last. Windows separated by a
  time gap (the person was not analyzed there) are not merged.
- Parallel actions ("A; B") go on separate rows. Rows of different actions can share the same
  time span, because the actions are parallel.
- Object columns: one per object type (the type is the id without its numeric suffix, so
  plate_1000 and plate_1001 go in "Plate"). People are objects too: the analyzed person is in the
  "Person" column of every row.
- Objects of an interval: the ids associated to the person in any of the merged windows. If
  there are several ids of the same type they go on separate rows (same Start, End and Action),
  one id per row in each type column.
"""
import os
import re
import csv
import glob

# PATHS
PATH_DATA_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT'
PATH_LOGS = os.path.join(PATH_DATA_ROOT, "action_recognition_logs")
PATH_LOGS_CONVERTED = os.path.join(PATH_DATA_ROOT, "action_recognition_logs_converted")

PERSON_SEPARATOR = ", "

_TIMESTAMP_RE = re.compile(r"^\s*([\d.]+)s?\s*-\s*([\d.]+)s?\s*$")
_ID_SUFFIX_RE = re.compile(r"[_\-\s]*\d+$")
_EPS = 1e-9


def object_type(object_id):
    """'plate_1000' -> 'Plate', 'cutting_board_7' -> 'Cutting board', 'person_1000' -> 'Person'."""
    name = _ID_SUFFIX_RE.sub("", object_id).replace("_", " ").strip() or object_id
    return name[:1].upper() + name[1:]


def parse_window(timestamp):
    """'0.00s-2.00s' -> (0.0, 2.0) seconds."""
    match = _TIMESTAMP_RE.match(timestamp)
    if not match:
        raise ValueError(f"Unexpected timestamp format: {timestamp!r}")
    return float(match.group(1)), float(match.group(2))


def convert_log(in_path, out_path):
    """Convert one log file. Returns (window rows read, action rows written)."""
    with open(in_path, newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    if not rows:
        return 0, 0

    header = rows[0]
    if len(header) < 3 or header[0] != "Timestamp" or header[-1] != "Action":
        raise ValueError(f"{in_path}: unexpected header {header}")
    object_ids = header[1:-1]
    # People first, then the other types in alphabetical order
    types = sorted({object_type(o) for o in object_ids}, key=lambda t: (t != "Person", t.lower()))

    # One entry per (window, person). If a video was re-run, the same window can appear more than
    # once in the log (rows are appended): the last one is kept.
    latest = {}
    n_read = 0
    for row in rows[1:]:
        if not row or all(not cell.strip() for cell in row):
            continue
        n_read += 1
        if len(row) != len(header):
            print(f"  [Warning] {os.path.basename(in_path)}: row with {len(row)} columns instead of {len(header)}, skipped")
            continue
        timestamp, flags, action = row[0], row[1:-1], row[-1]
        present = [oid for oid, flag in zip(object_ids, flags) if flag.strip() == "1"]
        persons = tuple(sorted(o for o in present if object_type(o) == "Person"))
        latest[(timestamp, persons)] = (present, action)

    windows_by_person = {}
    for (timestamp, persons), (present, action) in latest.items():
        start, end = parse_window(timestamp)
        windows_by_person.setdefault(persons, []).append({
            "start": start, "end": end, "center": (start + end) / 2,
            "objects": [o for o in present if object_type(o) != "Person"],
            "actions": list(dict.fromkeys(a.strip() for a in action.split(";") if a.strip())) or [""],
        })

    # Merge successive windows with the same action into intervals (one list of runs per person)
    runs = []
    for persons, windows in windows_by_person.items():
        windows.sort(key=lambda w: w["center"])
        for i, w in enumerate(windows):
            # time owned by this window: up to the midpoint with the neighbours, never outside the window
            w["seg_start"] = w["start"]
            w["seg_end"] = w["end"]
            if i > 0:
                w["seg_start"] = max(w["start"], (windows[i - 1]["center"] + w["center"]) / 2)
            if i + 1 < len(windows):
                w["seg_end"] = min(w["end"], (w["center"] + windows[i + 1]["center"]) / 2)

        open_runs = {}
        for w in windows:
            new_open = {}
            for action in w["actions"]:
                run = open_runs.get(action)
                if run is not None and abs(run["end"] - w["seg_start"]) < _EPS:
                    run["end"] = w["seg_end"]          # same action in the next, contiguous window
                    run["objects"].update(w["objects"])
                else:
                    run = {"start": w["seg_start"], "end": w["seg_end"], "persons": persons,
                           "action": action, "objects": set(w["objects"])}
                    runs.append(run)
                new_open[action] = run
            open_runs = new_open

    runs.sort(key=lambda r: (r["start"], r["persons"]))  # stable: action order inside a window is kept

    out_rows = []
    for run in runs:
        ids_by_type = {t: [] for t in types}
        for oid in sorted(run["objects"]):
            ids_by_type[object_type(oid)].append(oid)
        person_cell = PERSON_SEPARATOR.join(run["persons"])
        n_rows = max([1] + [len(ids) for t, ids in ids_by_type.items() if t != "Person"])
        for k in range(n_rows):  # several objects of the same type -> several rows
            cells = [person_cell if t == "Person" else (ids_by_type[t][k] if k < len(ids_by_type[t]) else "")
                     for t in types]
            out_rows.append([f"{run['start']:.2f}", f"{run['end']:.2f}"] + cells + [run["action"]])

    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Start", "End"] + types + ["Action"])
        writer.writerows(out_rows)
    return n_read, len(out_rows)


def convert_directory(in_dir, out_dir):
    log_files = sorted(glob.glob(os.path.join(in_dir, "*_log.csv")))
    if not log_files:
        print(f"No *_log.csv file found in {in_dir}")
        return 0
    for path in log_files:
        out_path = os.path.join(out_dir, os.path.basename(path))
        n_in, n_out = convert_log(path, out_path)
        print(f"[+] {os.path.basename(path)}: {n_in} window rows -> {n_out} action rows")
    print(f"Converted logs saved in {out_dir}")
    return len(log_files)


if __name__ == "__main__":
    convert_directory(PATH_LOGS, PATH_LOGS_CONVERTED)